# B1 — Cadrer et mesurer

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| B — Influence dans un réseau social | 1 sur 4 | 2 h 30 | M1, M4, M7 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Au début des années 1970, l'anthropologue Wayne Zachary a observé pendant deux ans un club de karaté d'une université américaine. Il a noté, pour chaque paire de membres, s'ils se fréquentaient **en dehors** du club, et dans combien de contextes (cours, bar du campus, tournois…). Pendant l'étude, un conflit a opposé le professeur (« Mr. Hi ») et le président du club (« John A. »), jusqu'à ce que le club se scinde en deux.

Ce réseau de 34 membres est devenu l'un des plus étudiés au monde. La question de ce projet paraît simple :

**Qui sont les membres les plus influents du réseau, et selon quel critère ?**

« Influent » n'a pas de définition unique. Ce projet en construit plusieurs, les compare, les met à l'épreuve sur un phénomène réel (la scission du club) et sur d'autres réseaux.

![Schéma : les quatre étapes du projet B, de B1 cadrer et mesurer à B4 généraliser et convaincre, avec les modules réinvestis](../../../assets/schemas/schema-projet-b-etapes.svg)

**Activité** : Proposez au moins trois sens différents du mot « influent » dans un groupe d'amis. Pour chacun, formulez une question précise à laquelle un calcul sur le réseau pourrait répondre.

Indiquez vos réponses et justifiez-les.

**Activité** : Quelles hypothèses fait-on en représentant un groupe humain par un réseau de liens ? Pensez à la nature des liens (symétriques ? tous de même force ?), au temps, aux membres absents des données.

Indiquez vos réponses et justifiez-les.

### La fiabilité de la source

Les données proviennent d'un seul observateur, il y a cinquante ans. Elles ont été recopiées de nombreuses fois : selon les versions, le réseau compte 77 ou 78 liens. Les fichiers du projet viennent de la bibliothèque `networkx` (78 liens, avec des poids).

**Activité** : Quelles questions poseriez-vous pour juger de la fiabilité de ces données ? Quelles conclusions de ce projet pourraient dépendre d'un lien de plus ou de moins ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reseaux import charger, dessiner

membres, liens, A = charger("karate")
fig, ax = plt.subplots(figsize=(7, 6))
dessiner(A, membres, ax=ax, titre="Le club de karaté de Zachary (taille uniforme)")
plt.tight_layout(); plt.show()

In [ ]:
pari_membre_le_plus_influent = 1        # numéro du membre (modifiez)
pari_criteres_d_accord = "oui"          # les différents critères désigneront-ils les mêmes membres ? (modifiez)

## ③ Abstraire : un réseau est une matrice

On numérote les membres de 1 à $n$. La **matrice d'adjacence** $A$ contient $A_{ij} = 1$ si les membres $i$ et $j$ sont liés, 0 sinon. Elle est symétrique.

![Schéma : un réseau de cinq membres et sa matrice d'adjacence ; le produit de A par le vecteur de 1 donne les degrés, et A au carré compte les chemins de longueur 2](../../../assets/schemas/schema-reseau-matrice.svg)

Les produits de matrices du module M3 prennent alors un sens concret :

- $A\mathbf{1}$ (somme de chaque ligne) donne le **degré** de chaque membre : son nombre de liens ;
- $(A^2)_{ij} = \sum_k A_{ik}A_{kj}$ compte les membres $k$ liés à la fois à $i$ et à $j$ : le nombre de **chemins de longueur 2** de $i$ à $j$ ;
- plus généralement, $(A^k)_{ij}$ compte les chemins de longueur $k$.

**Activité** : Sur le schéma, calculez à la main $(A^2)_{15}$ et $(A^2)_{23}$. Interprétez chaque nombre. Que vaut $(A^2)_{ii}$ ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Le degré : compter ses liens

In [ ]:
n = len(membres)
print(f"{n} membres, {int(A.sum() / 2)} liens ; A symétrique ? {np.allclose(A, A.T)}")

degre = A @ np.ones(n)
_, _, A_poids = charger("karate", pondere=True)
force = A_poids @ np.ones(n)                       # degré pondéré : somme des poids des liens

mesures = pd.DataFrame({"membre": membres["id"], "rôle": membres["role"].fillna(""),
                        "degré": degre.astype(int), "force": force.astype(int)})
print(mesures.sort_values("degré", ascending=False).head(8).to_string(index=False))

### Les voisins de voisins

In [ ]:
A2 = A @ A
print("Diagonale de A² = degrés ?", np.allclose(np.diag(A2), degre))

atteints_en_2 = ((A + A2) > 0).sum(axis=1) - 1          # membres à distance 1 ou 2 (soi-même exclu)
amis_communs = A2.copy()
np.fill_diagonal(amis_communs, 0)
i, j = np.unravel_index(np.argmax(amis_communs), amis_communs.shape)
print(f"Plus grand nombre d'amis communs : {int(amis_communs[i, j])}, entre les membres {membres['id'][i]} et {membres['id'][j]}")

mesures["atteints en 2 pas"] = atteints_en_2
print(mesures.sort_values("atteints en 2 pas", ascending=False).head(8).to_string(index=False))

### Les critères sont-ils d'accord ?

On compare les classements obtenus par chaque critère avec la **corrélation des rangs** : 1 si les deux classements sont identiques, 0 s'ils n'ont aucun rapport.

In [ ]:
criteres = ["degré", "force", "atteints en 2 pas"]
print(mesures[criteres].rank().corr().round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, c in zip(axes, criteres):
    dessiner(A, membres, mesures[c].to_numpy(), ax=ax, titre=c)
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : membre {pari_membre_le_plus_influent} ; critères d'accord : {pari_criteres_d_accord}")
for c in criteres:
    print(f"Premier selon « {c} » : membre {mesures.loc[mesures[c].idxmax(), 'membre']}")

**Activité** :

1. Qui sont les membres les plus influents selon ces premiers critères ? Que savez-vous de leur rôle dans le club ?
2. Le critère « atteints en 2 pas » distingue-t-il bien les membres ? Que se passe-t-il dans un petit réseau très connecté ?
3. Ces mesures ne regardent que les voisins proches. Quelle forme d'influence leur échappe ?
4. Rédigez, pour la suite du projet, une liste de sous-questions précises et l'ordre dans lequel vous les traiterez.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Pondérer ou non ?** Le degré et la force classent-ils les membres de la même façon ? Qu'est-ce qui justifierait de préférer l'un ou l'autre ?
- **Un lien de plus ou de moins.** Retirez au hasard un lien du réseau : le classement par degré change-t-il beaucoup ? Et le premier du classement ?
- **Généraliser : la proximité.** Un membre influent pourrait être celui qui est **proche** de tous les autres. La distance entre deux membres est la longueur du plus court chemin, c'est-à-dire le plus petit $k$ tel que $(A^k)_{ij} > 0$. Écrivez une fonction `distances(A)` qui calcule la matrice de toutes les distances avec des puissances de $A$, puis classez les membres selon leur distance moyenne aux autres.

In [ ]:
def distances(A):
    # À compléter : renvoie la matrice des distances (nombre de liens du plus court chemin)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def distances(A):
    n = len(A)
    D = np.full((n, n), np.inf)
    np.fill_diagonal(D, 0)
    puissance = np.eye(n)
    for k in range(1, n):
        puissance = puissance @ A
        nouveaux = (puissance > 0) & np.isinf(D)
        D[nouveaux] = k
        if not np.isinf(D).any():
            break
    return D

D = distances(A)
mesures["distance moyenne"] = D.sum(axis=1) / (n - 1)
print("Plus proches de tous les autres :")
print(mesures.sort_values("distance moyenne").head(5)[["membre", "rôle", "distance moyenne"]].round(2).to_string(index=False))
print("Diamètre du réseau (plus grande distance) :", int(D.max()))

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Découper un gros problème en petits problèmes": 1,
    "Formuler des questions précises": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Identifier les concepts pertinents et leurs relations": 1,
    "Qualifier la fiabilité des sources": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "B1-cadrer-et-mesurer", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")